# 004 — Первое реальное исследование (RUN-7)

Отчёт о первом прогоне платформы на настоящей истории Binance USDT-M futures.
Цель тикета — не доходность, а подтверждение, что пайплайн работает на живых
данных, и фиксация того, что на них разъехалось.

| | |
|---|---|
| Дата прогона | 2026-10-30 (UTC) |
| Конфиг | [`configs/experiments/ma_cross_multi_1h_4h.yaml`](../configs/experiments/ma_cross_multi_1h_4h.yaml) |
| Данные | BTCUSDT / ETHUSDT / SOLUSDT × 1h / 4h, 2022-01-01 … 2026-10-01 |
| Research-диапазон | 2022-01-01 … 2026-04-01 (выше — holdout) |
| Сетка | `fast_ma ∈ {10, 20, 50}` × `slow_ma ∈ {100, 200}` = 6 комбинаций |
| Слайсы | квартал, `min_bars=300` → 19 слайсов на пару, 114 пар×слайс |
| Железо | Apple Silicon, 14 ядер, Python 3.11 |

**Итог одной строкой:** все восемь шагов прошли без правок кода, кроме
`fetch-market-data`, который на историческом диапазоне падает целиком из-за
30-дневного окна Binance для OI/LS (см. раздел «Находки»). Результат
исследования: `slow_ma=200` устойчиво лучше `slow_ma=100` во всех разрезах,
но ни одна комбинация не даёт положительного `stability_score`, а holdout
подтвердил прибыль только на SOLUSDT 1h.

## Воспроизведение

Все команды — из корня репозитория, в активированном venv. Конфиг закоммичен,
данные (`data/`) вне git.

```bash
# 1. OHLCV — 3 символа × 2 таймфрейма (87 с, 156 060 свечей)
for s in BTCUSDT ETHUSDT SOLUSDT; do for tf in 1h 4h; do
  trading-research fetch-data --symbol $s --timeframe $tf \
    --start 2022-01-01 --end 2026-10-01
done; done

# 2. Рыночные ряды. funding — за всю историю; OI/LS Binance отдаёт только
#    за последние ~30 дней, поэтому они качаются отдельным окном (REG-4).
for s in BTCUSDT ETHUSDT SOLUSDT; do for tf in 1h 4h; do
  trading-research fetch-market-data --symbol $s --timeframe $tf \
    --start 2022-01-01 --end 2026-10-01 --series funding
  for ser in open_interest long_short_ratio; do
    trading-research fetch-market-data --symbol $s --timeframe $tf \
      --start 2026-10-02 --end 2026-10-30 --series $ser
  done
done; done

# 3. Фичи режима по слайсам
trading-research build-features --config configs/experiments/ma_cross_multi_1h_4h.yaml

# 4. Сетка по слайсам (4.7 с на 14 воркерах)
trading-research run-experiment --config configs/experiments/ma_cross_multi_1h_4h.yaml

# 5. Ранжирование и stability
trading-research analyze-results --experiment ma_cross_multi --metric sharpe --top 6
trading-research analyze-results --experiment ma_cross_multi --metric sharpe --by symbol --top 9
trading-research export-results --experiment ma_cross_multi --kind stability --metric sharpe

# 6. Карты параметров
trading-research plot-heatmap --experiment ma_cross_multi --x fast_ma --y slow_ma \
  --metric sharpe --agg median

# 7. Честный OOS + holdout (13 с)
trading-research walk-forward --config configs/experiments/ma_cross_multi_1h_4h.yaml

# 8. Корреляции параметры ↔ режим
trading-research analyze-regimes --experiment ma_cross_multi --metric sharpe
trading-research analyze-regimes --experiment ma_cross_multi --metric sharpe --global
trading-research analyze-regimes --experiment ma_cross_multi --metric sharpe --regime bull_trend
```

In [ ]:
import json
import os
from datetime import UTC, datetime
from pathlib import Path

import polars as pl

from trading_research.analytics.correlations import (
    attach_regime_features,
    best_params_in_regime,
    regime_correlations,
)
from trading_research.analytics.queries import load_performance_table, rank_params
from trading_research.analytics.stability import rank_by_stability, stability_vector
from trading_research.data.storage.parquet_store import ParquetStore
from trading_research.data.validators import validate_candles

EXPERIMENT = "ma_cross_multi"
SYMBOLS = ("BTCUSDT", "ETHUSDT", "SOLUSDT")
TIMEFRAMES = ("1h", "4h")
START, END = datetime(2022, 1, 1, tzinfo=UTC), datetime(2026, 10, 1, tzinfo=UTC)

RAW = Path(os.environ.get("TR_RAW", "data/raw"))
RESULTS = Path(os.environ.get("TR_RESULTS", "data/results")) / EXPERIMENT
FEATURES = Path(os.environ.get("TR_FEATURES", "data/features")) / EXPERIMENT

PERFORMANCE = RESULTS / "performance.parquet"
FEATURES_TABLE = FEATURES / "time_slice_features.parquet"

# Артефакты прогона лежат вне git: без них ноутбук читается как отчёт — числа и
# выводы в markdown-ячейках, — а код просто сообщает, чего не хватает.
HAVE_RAW = (RAW / "candles").exists()
HAVE_PERFORMANCE = PERFORMANCE.exists()
HAVE_FEATURES = FEATURES_TABLE.exists()
HAVE_WF = (RESULTS / "wf").exists()

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)


def skip(what: str, command: str) -> None:
    """Сообщить, какой шаг нужно прогнать, чтобы ячейка ожила."""
    print(f"нет артефактов: {what}\nпрогнать: {command}")


print(f"raw={HAVE_RAW} performance={HAVE_PERFORMANCE} features={HAVE_FEATURES} wf={HAVE_WF}")

## 1. Данные и их качество

Ожидание тикета было: `validators.py` (DATA-3) на реальной истории Binance
«почти наверняка найдёт пропуски свечей и дубли». **Не нашёл.** 156 060 свечей
без единого пропуска, дубля или нарушения OHLC-инвариантов: 41 616 часовых бар
на символ — это ровно число часов в `[2022-01-01, 2026-10-01)`, то есть история
полная.

Единственный «дефект», который репортит валидатор, — ровно один пропущенный бар
на правом краю у каждой пары, и это дефект валидатора, а не данных:
`validate_candles(expected_end=...)` считает границу **включительной**, а
`fetch-data --end` — **эксклюзивной**. Выписано в DATA-7.

In [ ]:
if not HAVE_RAW:
    skip("свечи", "trading-research fetch-data …  (см. раздел «Воспроизведение»)")
else:
    store = ParquetStore(RAW)
    rows = []
    for symbol in SYMBOLS:
        for tf in TIMEFRAMES:
            df = store.read_candles(
                exchange="binance",
                market="futures",
                symbol=symbol,
                timeframe=tf,
                start=START,
                end=END,
            )
            report = validate_candles(df, timeframe=tf, expected_start=START, expected_end=END)
            rows.append(
                {
                    "symbol": symbol,
                    "timeframe": tf,
                    "bars": report.total_bars,
                    "gaps": len(report.gaps),
                    "missing_bars": report.missing_bars_total,
                    "duplicates": len(report.duplicate_times),
                    "monotonic": report.is_monotonic,
                    "ohlc_bad": len(report.ohlc_inconsistencies),
                    "outliers": len(report.price_outliers),
                }
            )
    print(pl.DataFrame(rows))

### Рыночные ряды

| Ряд | Эндпоинт | Что получилось |
|---|---|---|
| funding | `/fapi/v1/fundingRate` | вся история: 5 202 точки на пару, 2022-01-01 … 2026-09-30 |
| open interest | `/futures/data/openInterestHist` | только `[сегодня − 30 дней]`, на research-диапазоне данных нет |
| long/short ratio | `/futures/data/globalLongShortAccountRatio` | то же |

Отсюда два следствия, оба зафиксированы тикетами:

- `fetch-market-data --series all` на историческом диапазоне **падает целиком**
  с сырым `Binance API error 400: parameter 'startTime' is invalid` — funding
  успевает записаться, OI и LS нет (REG-4);
- `oi_change_pct` и `ls_ratio_avg` в `time_slice_features` — 114 NULL из 114,
  а в матрице корреляций — `NaN`, и ни один шаг пайплайна об этом не
  предупреждает: «нет данных» выглядит как «нет корреляции» (REG-4).

Отдельно: запрос OI/LS окном шире `limit` **молча теряет начало диапазона** —
`/futures/data/*` отдаёт хвост окна, а пагинация двигает курсор вперёд от
`startTime` (REG-5). Из 28 запрошенных дней записались последние 21.

## 2. Сетка по слайсам

684 строки `parameter_performance_by_slice` = 6 комбинаций × 114 (пара × слайс),
684 записи в ledger — по одной на дедуплицированный run.

Результат ранжирования по `sharpe` (медиана по слайсам), глобально:

| ранг | params | sharpe | n |
|---|---|---|---|
| 1 | `fast_ma=50, slow_ma=200` | 0.471 | 114 |
| 2 | `fast_ma=10, slow_ma=200` | 0.436 | 114 |
| 3 | `fast_ma=20, slow_ma=200` | 0.393 | 114 |
| 4 | `fast_ma=10, slow_ma=100` | 0.192 | 114 |
| 5 | `fast_ma=50, slow_ma=100` | 0.110 | 114 |
| 6 | `fast_ma=20, slow_ma=100` | 0.096 | 114 |

**`slow_ma=200` лучше `slow_ma=100` при любом `fast_ma`** — разделение чистое,
все три комбинации с медленной MA 200 стоят выше всех трёх с MA 100. Это
единственный вывод исследования, который держится во всех разрезах: по символам,
по таймфреймам и по режимам (кроме `sideways_low_vol`).

При этом **`stability_score` отрицателен у всех шести комбинаций** (от −0.95 у
лучшей до −1.23 у худшей): разброс по окнам доминирует над медианой, доля
положительных слайсов 0.49…0.56, худший слайс −2.9…−4.7 sharpe. То есть
стабильной области параметров на этой сетке нет — по принципу №7 брать в бой
нечего.

In [ ]:
if not HAVE_PERFORMANCE:
    skip(
        "performance.parquet",
        "trading-research run-experiment --config configs/experiments/ma_cross_multi_1h_4h.yaml",
    )
else:
    performance = load_performance_table(PERFORMANCE)
    print(f"строк: {performance.height}, слайсов: {performance['time_slice_id'].n_unique()}")
    print(rank_params(performance, metric="sharpe", top=6))
    vector = stability_vector(performance, return_metric="sharpe")
    print(rank_by_stability(vector, top=6))

## 3. Walk-forward и holdout

45 фолдов на пару (train 180 д / test 30 д / step 30 д, скользящее окно),
270 фолдов всего, отбор на train по `sharpe`.

| пара | OOS return | OOS dd | OOS sharpe | trades | holdout params | holdout sharpe | деградация | reliable |
|---|---|---|---|---|---|---|---|---|
| BTCUSDT 1h | −4.55% | 35.06% | 0.070 | 306 | `50 / 100` | −1.704 | 1.774 | да |
| BTCUSDT 4h | +20.74% | 15.13% | 0.523 | 27 | `20 / 200` | +1.668 | −1.145 | **нет** (6 сделок) |
| ETHUSDT 1h | +19.42% | 46.66% | 0.310 | 306 | `20 / 100` | −1.466 | 1.776 | да |
| ETHUSDT 4h | −10.26% | 18.30% | −0.242 | 23 | `10 / 100` | +0.604 | −0.846 | да (16) |
| SOLUSDT 1h | +136.78% | 30.27% | 0.761 | 262 | `50 / 200` | +1.349 | −0.589 | да (30) |
| SOLUSDT 4h | −21.00% | 25.99% | −0.639 | 10 | `10 / 200` | +2.473 | −3.113 | **нет** (6 сделок) |

Что из этого следует:

1. **Единственный надёжный положительный holdout — SOLUSDT 1h** (+21.1% на 30
   сделках). Два лучших holdout-числа в таблице (SOL 4h sharpe 2.47, BTC 4h 1.67)
   получены на **6 сделках** и помечены `reliable=false` — на них нельзя
   опираться, и именно для этого в метриках есть `min_trades`.
2. **На 1h честный OOS оптимистичен**: у BTC и ETH research-OOS около нуля и
   выше, а holdout — −1.70 и −1.47 sharpe. Деградация ≈ 1.77 у обоих, то есть
   даже walk-forward не поймал смену режима в резерве.
3. **Выбор параметров по фолдам нестабилен**: ни один набор не выигрывает больше
   чем в 20 из 45 фолдов. На BTC 1h чаще всего (19 из 45) выбирался
   `50 / 100` — комбинация, которая в research-ранжировании стоит **пятой из
   шести**. Holdout считается по выбору последнего фолда (`_holdout_params`),
   поэтому BTC 1h проверялся именно на ней.
4. `walk-forward` печатает только OOS: holdout считается и сохраняется в
   `holdout_report.json`, но в вывод не попадает ни строкой — главный результат
   исследования приходится читать из файлов (RUN-9).

In [ ]:
if not HAVE_WF:
    skip(
        "артефакты walk-forward",
        "trading-research walk-forward --config configs/experiments/ma_cross_multi_1h_4h.yaml",
    )
else:
    rows = []
    for path in sorted((RESULTS / "wf").glob("*/*/wf_selections.json")):
        symbol, timeframe = path.parts[-3], path.parts[-2]
        payload = json.loads(path.read_text(encoding="utf-8"))
        oos = payload["oos_metrics"]
        report_path = path.with_name("holdout_report.json")
        holdout = (
            json.loads(report_path.read_text(encoding="utf-8")) if report_path.exists() else {}
        )
        rows.append(
            {
                "symbol": symbol,
                "timeframe": timeframe,
                "folds": len(payload["selections"]),
                "oos_return_pct": oos["total_return_pct"],
                "oos_sharpe": oos["sharpe"],
                "oos_trades": oos["trades_count"],
                "holdout_params": json.dumps(holdout.get("params", {}), sort_keys=True),
                "holdout_value": holdout.get("holdout_value"),
                "degradation": holdout.get("degradation"),
                "holdout_trades": holdout.get("holdout_metrics", {}).get("trades_count"),
                "holdout_reliable": holdout.get("holdout_metrics", {}).get("reliable"),
            }
        )
    print(pl.DataFrame(rows))

## 4. Режимы

19 кварталов × 6 пар дали такое распределение режимов: `recovery` 44,
`bear_trend` 25, `panic_dump` 20, `sideways_high_vol` 10, `sideways_low_vol` 9,
`bull_trend` 6. Выборка перекошена: на `bull_trend` приходится 6 слайсов из 114,
поэтому «лучшие параметры в bull_trend» — это утверждение о шести кварталах.

Глобальные корреляции `sharpe` ↔ фичи режима:

| фича | corr |
|---|---|
| market_return_pct | 0.343 |
| trend_strength | 0.258 |
| volatility | 0.171 |
| adx | 0.102 |
| volume_zscore | 0.077 |
| funding_avg | 0.051 |
| drawdown_from_ath | −0.033 |
| oi_change_pct | **NaN** (нет данных) |
| ls_ratio_avg | **NaN** (нет данных) |

Ожидаемо и скучно: MA Cross — трендовая стратегия, и её результат коррелирует
прежде всего с доходностью рынка (0.34) и силой тренда (0.26). Знак и порядок
одинаковы у всех шести комбинаций — то есть это свойство стратегии, а не
параметров.

По режимам: в `bull_trend` лучшие — `50 / 200` (sharpe 1.02) и `10 / 200`
(0.99); в `bear_trend` **все шесть комбинаций убыточны** (от −0.04 до −0.88),
лучшая — та же `50 / 200`; в `sideways_low_vol` порядок меняется — выигрывает
`10 / 200` и `10 / 100`, а `50 / 200` уходит в минус. Это единственный разрез,
где `slow_ma=200` не доминирует.

In [ ]:
if not (HAVE_PERFORMANCE and HAVE_FEATURES):
    skip(
        "performance.parquet + time_slice_features.parquet",
        "trading-research build-features --config configs/experiments/ma_cross_multi_1h_4h.yaml",
    )
else:
    performance = load_performance_table(PERFORMANCE)
    features = pl.read_parquet(FEATURES_TABLE)
    joined = attach_regime_features(performance, features)
    print(features.group_by("regime_label").len().sort("len", descending=True))
    print(regime_correlations(joined, metric="sharpe", by=None))
    for regime in ("bull_trend", "bear_trend", "sideways_low_vol"):
        print(f"\n{regime}:")
        print(best_params_in_regime(joined, regime_label=regime, metric="sharpe", top=6))

## 5. Замеры — вход для RUN-8

| Шаг | Время | Пиковая RSS | Объём |
|---|---|---|---|
| `fetch-data` × 6 пар | 87 с | — | 156 060 свечей, 9.1 МБ parquet |
| `fetch-market-data` funding × 6 | 35 с | — | 31 212 точек, 1.3 МБ |
| `build-features` | 1 с | — | 114 строк, 12 КБ |
| `run-experiment`, `--workers 14` | **4.7 с** | 184 МБ (родитель) | 36 непрерывных прогонов → 684 строки |
| `run-experiment`, `--workers 1` | **9.7 с** | 226 МБ | то же |
| `walk-forward` | **13.0 с** | 143 МБ | 270 фолдов × 6 комбинаций = 1 620 train-бэктестов + 270 test |
| `analyze-*`, `plot-heatmap` | < 2 с каждый | — | — |
| `data/` итого | — | — | 10.5 МБ raw, 20 МБ results |

Что эти числа говорят про находки RUN-8:

- **Параллелизм почти не работает**: 14 воркеров дают ускорение всего **2.07×**
  (9.7 → 4.7 с), при этом user-time растёт с 9.4 до 17.6 с. Накладные расходы на
  сериализацию съедают больше половины выигрыша — на этом размере сетки.
- **Walk-forward однопоточный и уже доминирует**: 13.0 с real при 13.1 с user,
  то есть находка №1 RUN-8 (`_select_on_train` последовательный) — это ровно
  весь бюджет шага. Сетка из 6 комбинаций и 45 фолдов на пару — профиль, на
  котором параллелить ещё нечего; разница появится на сетке в сотни комбинаций.
- Масштаб для бенчмарка RUN-8: **1 620 train-бэктестов по ~41 600 бар** на пару
  1h. Память не является ограничением (< 250 МБ на всё), поэтому оптимизировать
  нужно время, а не аллокации.

## 6. Находки → тикеты

Прогон не вскрыл ни одной ошибки в счётной части: движок, метрики, нарезка
слайсов, walk-forward и holdout на реальных данных отработали без правок. Всё
найденное — на стыке с внешним источником и в выводе результатов.

| Тикет | Что не так | Почему важно |
|---|---|---|
| [REG-4](../tickets/REG-4.md) | OI/LS доступны только за ~30 дней; `--series all` падает сырой ошибкой API, а пустые фичи выглядят как «корреляции нет» | две из девяти фич режима молча пусты на любом историческом прогоне |
| [REG-5](../tickets/REG-5.md) | пагинация `/futures/data/*` молча теряет начало диапазона (эндпоинт отдаёт хвост окна) | тихая потеря данных: записалось 500 строк из 672, без ошибки |
| [DATA-7](../tickets/DATA-7.md) | `validate_candles(expected_end=…)` включителен, `fetch-data --end` эксклюзивен | валидатор на чистых данных всегда репортит один ложный пропуск |
| [DATA-8](../tickets/DATA-8.md) | DATA-3 недоступен из CLI — валидацию пришлось запускать скриптом | тот же класс разрыва «библиотека есть, в CLI не подключено», что закрывали RUN-1…RUN-5 |
| [RUN-9](../tickets/RUN-9.md) | `walk-forward` не печатает holdout | главный результат исследования читается только из JSON |

Отдельно — **не** баг: режимные слоты в `performance.parquet` остаются `null`.
Так решено в RUN-5: их заполняет join на этапе анализа, поэтому `run-experiment`
и `build-features` не имеют порядковой зависимости.